In [ ]:
# Parameters
num_bits = 10


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
import matplotlib.pyplot as plt
import numpy as np

n_bits = int(num_bits)

print(f"BB84 Quantum Key Distribution — Bits: {n_bits}")

# 1. Alice generates random bits and random bases
alice_bits = np.random.randint(2, size=n_bits)
alice_bases = np.random.randint(2, size=n_bits) 

qc = QuantumCircuit(n_bits, n_bits)
for i in range(n_bits):
    if alice_bits[i] == 1:
        qc.x(i) 
    if alice_bases[i] == 1:
        qc.h(i)
qc.barrier()

# 2. Bob guesses random bases
bob_bases = np.random.randint(2, size=n_bits)

for i in range(n_bits):
    if bob_bases[i] == 1:
        qc.h(i) 
    qc.measure(i, i)

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
job = simulator.run(transpile(qc, simulator), shots=1)
# Results come back as a string, e.g., '10101...' 
bob_results = list(job.result().get_counts().keys())[0][::-1]
bob_bits = [int(b) for b in bob_results]

sifted_key = []
for i in range(n_bits):
    if alice_bases[i] == bob_bases[i]:
        sifted_key.append(bob_bits[i])

print(f"Alice's Bases: {list(alice_bases)}")
print(f"Bob's Bases:   {list(bob_bases)}")
print(f"Matches found: {len(sifted_key)}")
print(f"Secret Key:    {sifted_key}")

counts = {'Matches': len(sifted_key), 'Mismatches': n_bits - len(sifted_key)}
fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere (show first qubit)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={np.pi if alice_bits[0] == 1 else 0:.6f}")
print(f"BLOCH_PHI={0.0:.6f}")
